In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.6 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import math
import os
from collections import defaultdict
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

model = YOLO("/content/drive/MyDrive/YoloModels/best.pt")

input_folder = "/content/drive/MyDrive/Yolov8_Input_Videos"
output_folder = "/content/drive/MyDrive/Yolov8_Videos_Results"

os.makedirs(output_folder, exist_ok=True)

video_files = [f for f in os.listdir(input_folder) if f.endswith((".mp4",".avi",".mov"))]

print("Videos found:", video_files)

for video_file in video_files:

    video_path = os.path.join(input_folder, video_file)

    output_path = os.path.join(
        output_folder,
        f"output_{video_file}"
    )

    heatmap_path = os.path.join(
        output_folder,
        f"heatmap_{video_file}.png"
    )

    print("\nProcessing:", video_file)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Could not open video:", video_file)
        continue

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps == 0:
        fps = 25

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    previous_positions = {}
    previous_times = {}
    inactive_frames = defaultdict(int)
    species_ids = defaultdict(set)

    heatmap = np.zeros((height, width), dtype=np.float32)

    frame_number = 0

    ACTIVE_THRESHOLD = 10
    INACTIVE_SECONDS = 10

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        frame_number += 1
        current_time = frame_number / fps

        results = model.track(frame, persist=True, conf=0.4)

        centers = []

        if results[0].boxes.id is not None:

            boxes = results[0].boxes
            ids = boxes.id.int().tolist()
            classes = boxes.cls.int().tolist()

            for box, track_id, cls in zip(boxes.xyxy, ids, classes):

                x1, y1, x2, y2 = map(int, box.tolist())
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                centers.append((track_id, cx, cy))

                class_name = model.names[cls]
                species_ids[class_name].add(track_id)

                speed = 0

                if track_id in previous_positions:

                    px, py = previous_positions[track_id]
                    pt = previous_times[track_id]

                    distance = math.sqrt((cx - px)**2 + (cy - py)**2)
                    time_diff = current_time - pt

                    if time_diff > 0:
                        speed = distance / time_diff

                previous_positions[track_id] = (cx, cy)
                previous_times[track_id] = current_time

                if speed > ACTIVE_THRESHOLD:
                    state = "Active"
                    inactive_frames[track_id] = 0
                else:
                    state = "Idle"
                    inactive_frames[track_id] += 1

                alert = ""

                if inactive_frames[track_id] > fps * INACTIVE_SECONDS:
                    alert = "Inactive Too Long"

                if 0 <= cy < height and 0 <= cx < width:
                    heatmap[cy, cx] += 1

                color = (0,255,0) if state == "Active" else (0,0,255)

                cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)

                label = f"{class_name} ID:{track_id} {state}"

                font_scale = 1.2
                thickness = 3

                (text_width,text_height),_ = cv2.getTextSize(
                    label,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale,
                    thickness
                )

                cv2.rectangle(
                    frame,
                    (x1,y1-text_height-15),
                    (x1+text_width+10,y1),
                    (0,0,0),
                    -1
                )

                cv2.putText(
                    frame,
                    label,
                    (x1+5,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale,
                    color,
                    thickness
                )

                if alert:
                    cv2.putText(
                        frame,
                        alert,
                        (x1,y2+30),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.9,
                        (0,0,255),
                        3
                    )

        out.write(frame)

    cap.release()
    out.release()

    heatmap += 1e-6

    heatmap_blur = cv2.GaussianBlur(heatmap,(35,35),0)
    heatmap_norm = cv2.normalize(heatmap_blur,None,0,255,cv2.NORM_MINMAX)
    heatmap_uint8 = heatmap_norm.astype(np.uint8)
    heatmap_color = cv2.applyColorMap(heatmap_uint8,cv2.COLORMAP_JET)

    cv2.imwrite(heatmap_path,heatmap_color)

    print("Saved:", output_path)

print("\nAll videos processed successfully.")

Streaming output truncated to the last 5000 lines.
0: 640x448 1 Lion, 278.1ms
Speed: 12.3ms preprocess, 278.1ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 448)

0: 640x448 1 Lion, 289.3ms
Speed: 19.3ms preprocess, 289.3ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 448)

0: 640x448 1 Lion, 256.0ms
Speed: 11.1ms preprocess, 256.0ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 448)

0: 640x448 1 Lion, 190.2ms
Speed: 7.7ms preprocess, 190.2ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 448)

0: 640x448 1 Lion, 177.4ms
Speed: 5.9ms preprocess, 177.4ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 448)
Saved: /content/drive/MyDrive/Yolov8_Videos_Results/output_Test3.mp4

Processing: Test5.mp4

0: 384x640 1 Tiger, 146.4ms
Speed: 5.3ms preprocess, 146.4ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Tiger, 144.4ms
Speed: 10.4ms preprocess, 144.4ms inference, 0.9ms postprocess per im